In [1]:
# Cell 1 — Imports + helpers S3

import numpy as np
import pandas as pd
import pyarrow.parquet as pq
import pyarrow.fs as pafs
from urllib.parse import urlparse

def read_parquet_s3(path: str) -> pd.DataFrame:
    assert path.startswith("s3://")
    u = urlparse(path)
    fs = pafs.S3FileSystem()
    s3_path = f"{u.netloc}/{u.path.lstrip('/')}"
    with fs.open_input_file(s3_path) as f:
        return pq.read_table(f).to_pandas()

S3_PATH = "s3://tradebot-config-tokyo/data/s1/events_all_base.parquet"

In [2]:
# Cell 2 — Load + aperçu
df = read_parquet_s3(S3_PATH)
print("rows:", len(df), "cols:", df.shape[1])
df.head(3)

rows: 39812 cols: 34


,t0_close,dir,R_bps,close,c_absret_15m,c_absret_30m,c_absret_1h,c_vol_30m,c_vol_1h,c_range_30m_bps,...,bk_spread_bps,bk_micro_diff_bps,bk_imb_L1,t0_high,t0_low,t0,symbol,decision_freq_min,lookback_bars,side
0,42649.1,1,NaN,42649.1,11.457743,8.074496,7.361476,10.700380,9.712657,43.846177,...,0.023475,-0.003041,-0.259064,42659.4,42598.2,2024-01-01 01:35:00+00:00,BTCUSDT,5,12,long
1,42739.1,1,NaN,42739.1,17.222355,11.513093,7.466947,11.539879,9.894005,65.116018,...,0.023385,-0.001788,-0.152879,42775.8,42739.1,2024-01-01 01:40:00+00:00,BTCUSDT,5,12,long
2,42562.7,-1,NaN,42562.7,6.163115,5.640304,5.546644,6.771490,5.946680,26.877994,...,0.023504,-0.008725,-0.742381,42562.8,42530.0,2024-01-01 02:50:00+00:00,BTCUSDT,5,12,short


In [3]:
# Cell 3 — Schéma / dtypes / mémoire
display(df.dtypes.value_counts())
print("\nTop 30 colonnes par 'non-null':")
display(df.notna().sum().sort_values(ascending=False).head(30))

print("\nTop 30 colonnes par NaN:")
display(df.isna().sum().sort_values(ascending=False).head(30))

float64                28
object                  2
int64                   2
int8                    1
datetime64[ns, UTC]     1
Name: count, dtype: int64


Top 30 colonnes par 'non-null':


t0_close             39812
bk_micro_diff_bps    39812
bk_bid0              39812
bk_ask0              39812
bk_mid               39812
bk_bid0_sz           39812
bk_ask0_sz           39812
bk_spread_bps        39812
bk_imb_L1            39812
dir                  39812
t0_high              39812
t0_low               39812
t0                   39812
symbol               39812
decision_freq_min    39812
lookback_bars        39812
tr_aggr_buy_ratio    39812
tr_signed_vol        39812
tr_vwap              39812
tr_vol               39812
tr_count             39812
c_compression        39812
c_range_1h_bps       39812
c_range_30m_bps      39812
c_vol_1h             39812
c_vol_30m            39812
c_absret_1h          39812
c_absret_30m         39812
c_absret_15m         39812
close                39812
dtype: int64


Top 30 colonnes par NaN:


R_bps                38
c_trend_1h_bps        5
c_trend_30m_bps       1
t0_close              0
bk_micro_diff_bps     0
bk_mid                0
bk_bid0_sz            0
bk_ask0_sz            0
bk_spread_bps         0
bk_imb_L1             0
bk_bid0               0
t0_high               0
t0_low                0
t0                    0
symbol                0
decision_freq_min     0
lookback_bars         0
bk_ask0               0
tr_signed_vol         0
tr_aggr_buy_ratio     0
dir                   0
tr_vwap               0
tr_vol                0
tr_count              0
c_compression         0
c_range_1h_bps        0
c_range_30m_bps       0
c_vol_1h              0
c_vol_30m             0
c_absret_1h           0
dtype: int64

In [4]:
# Cell 4 — Vérifier le temps : colonne t0 ou index
has_t0_col = "t0" in df.columns
print("has t0 column:", has_t0_col)

if has_t0_col:
    t0 = pd.to_datetime(df["t0"], utc=True, errors="coerce")
    print("t0 parse ok:", t0.notna().mean())
    print("t0 min/max:", t0.min(), "->", t0.max())
else:
    # tentative index
    idx = pd.to_datetime(df.index, utc=True, errors="coerce")
    print("index parse ok:", idx.notna().mean())
    print("index min/max:", idx.min(), "->", idx.max())

has t0 column: True
t0 parse ok: 1.0
t0 min/max: 2024-01-01 01:35:00+00:00 -> 2025-10-31 23:15:00+00:00


In [5]:
# Cell 5 — Construire un df “indexé temps” comme le fera expertB_light
df2 = df.copy()

if "t0" in df2.columns:
    df2["t0"] = pd.to_datetime(df2["t0"], utc=True, errors="coerce")
    df2 = df2.dropna(subset=["t0"]).sort_values("t0").set_index("t0")
else:
    df2.index = pd.to_datetime(df2.index, utc=True, errors="coerce")
    df2 = df2.dropna(axis=0, subset=[]).sort_index()

print("after time-indexing => rows:", len(df2), "cols:", df2.shape[1])
print("index tz:", getattr(df2.index, "tz", None))
print("monotonic:", df2.index.is_monotonic_increasing)
print("min/max:", df2.index.min(), "->", df2.index.max())
df2.head(3)


after time-indexing => rows: 39812 cols: 33
index tz: UTC
monotonic: True
min/max: 2024-01-01 01:35:00+00:00 -> 2025-10-31 23:15:00+00:00


,t0_close,dir,R_bps,close,c_absret_15m,c_absret_30m,c_absret_1h,c_vol_30m,c_vol_1h,c_range_30m_bps,...,bk_ask0_sz,bk_spread_bps,bk_micro_diff_bps,bk_imb_L1,t0_high,t0_low,symbol,decision_freq_min,lookback_bars,side
t0,,,,,,,,,,,,,,,,,,,,,
2024-01-01 01:35:00+00:00,42649.1,1,NaN,42649.1,11.457743,8.074496,7.361476,10.700380,9.712657,43.846177,...,8.595,0.023475,-0.003041,-0.259064,42659.4,42598.2,BTCUSDT,5,12,long
2024-01-01 01:40:00+00:00,42739.1,1,NaN,42739.1,17.222355,11.513093,7.466947,11.539879,9.894005,65.116018,...,3.254,0.023385,-0.001788,-0.152879,42775.8,42739.1,BTCUSDT,5,12,long
2024-01-01 02:50:00+00:00,42562.7,-1,NaN,42562.7,6.163115,5.640304,5.546644,6.771490,5.946680,26.877994,...,17.524,0.023504,-0.008725,-0.742381,42562.8,42530.0,BTCUSDT,5,12,short


In [6]:
# Cell 6 — Colonnes requises pour make_events_expertB_light.py
required = ["dir", "R_bps", "t0_close"]
missing = [c for c in required if c not in df2.columns]
print("missing required:", missing)

# vérif types + valeurs
for c in required:
    if c in df2.columns:
        print("\n", c)
        print("dtype:", df2[c].dtype)
        print("non-null:", df2[c].notna().mean())
        print("min/max:", np.nanmin(df2[c].to_numpy(dtype=float)), np.nanmax(df2[c].to_numpy(dtype=float)))
        

missing required: []

 dir
dtype: int8
non-null: 1.0
min/max: -1.0 1.0

 R_bps
dtype: float64
non-null: 0.9990455139154024
min/max: 12.67924123555633 386.50316081133064

 t0_close
dtype: float64
non-null: 1.0
min/max: 38622.1 126070.9


In [7]:
# Cell 7 — Sanity checks “métier” (dir ∈ {-1,+1}, R_bps > 0, t0_close > 0)
checks = {}

if "dir" in df2.columns:
    vals = pd.Series(df2["dir"]).dropna().astype(int)
    checks["dir_unique_values"] = sorted(vals.unique().tolist())
    checks["dir_ok_rate_(in_-1_+1)"] = float(vals.isin([-1, 1]).mean())

if "R_bps" in df2.columns:
    r = pd.to_numeric(df2["R_bps"], errors="coerce")
    checks["R_bps_nonnull_rate"] = float(r.notna().mean())
    checks["R_bps_pos_rate"] = float((r > 0).mean())
    checks["R_bps_p50"] = float(np.nanmedian(r.to_numpy(dtype=float)))

if "t0_close" in df2.columns:
    px = pd.to_numeric(df2["t0_close"], errors="coerce")
    checks["t0_close_nonnull_rate"] = float(px.notna().mean())
    checks["t0_close_pos_rate"] = float((px > 0).mean())

checks

{'dir_unique_values': [-1, 1],
 'dir_ok_rate_(in_-1_+1)': 1.0,
 'R_bps_nonnull_rate': 0.9990455139154024,
 'R_bps_pos_rate': 0.9990455139154024,
 'R_bps_p50': 62.954893521017,
 't0_close_nonnull_rate': 1.0,
 't0_close_pos_rate': 1.0}

In [8]:
# Cell 8 — Doublons temps (important : 1 event par t0)
dup = df2.index.duplicated().sum()
print("duplicated t0:", dup)

if dup > 0:
    display(df2.index[df2.index.duplicated()].value_counts().head(10))

duplicated t0: 0


In [9]:
# Cell 9 — Vérifier qu’il y a des features numériques exploitables
# Colonnes numériques
num_cols = [c for c in df2.columns if pd.api.types.is_numeric_dtype(df2[c])]
print("numeric cols:", len(num_cols))

# Candidats à exclure (label future/leakage si un jour présents)
leak_cols = [c for c in ["mfe_R", "mae_R", "outcome", "y", "label", "pB", "tp_rr", "max_pullback_R"] if c in df2.columns]
print("potential leakage/targets present:", leak_cols)

# un aperçu
display(pd.Series(num_cols).head(40))

numeric cols: 31
potential leakage/targets present: []


0              t0_close
1                   dir
2                 R_bps
3                 close
4          c_absret_15m
5          c_absret_30m
6           c_absret_1h
7             c_vol_30m
8              c_vol_1h
9       c_range_30m_bps
10       c_range_1h_bps
11        c_compression
12      c_trend_30m_bps
13       c_trend_1h_bps
14             tr_count
15               tr_vol
16              tr_vwap
17        tr_signed_vol
18    tr_aggr_buy_ratio
19              bk_bid0
20              bk_ask0
21               bk_mid
22           bk_bid0_sz
23           bk_ask0_sz
24        bk_spread_bps
25    bk_micro_diff_bps
26            bk_imb_L1
27              t0_high
28               t0_low
29    decision_freq_min
30        lookback_bars
dtype: object

In [10]:
# Cell 10 — Vérifier la “qualité” des features : NaN / inf
X = df2[num_cols].copy()

# inf -> NaN
X = X.replace([np.inf, -np.inf], np.nan)

nan_rate = X.isna().mean().sort_values(ascending=False)
print("Top 30 NaN-rate features:")
display(nan_rate.head(30))

print("\nFeatures avec > 80% NaN:")
display(nan_rate[nan_rate > 0.80].head(50))

Top 30 NaN-rate features:


R_bps                0.000954
c_trend_1h_bps       0.000126
c_trend_30m_bps      0.000025
t0_close             0.000000
bk_ask0_sz           0.000000
bk_bid0              0.000000
bk_ask0              0.000000
bk_mid               0.000000
bk_bid0_sz           0.000000
bk_spread_bps        0.000000
tr_signed_vol        0.000000
bk_micro_diff_bps    0.000000
bk_imb_L1            0.000000
t0_high              0.000000
t0_low               0.000000
decision_freq_min    0.000000
tr_aggr_buy_ratio    0.000000
tr_vol               0.000000
tr_vwap              0.000000
dir                  0.000000
tr_count             0.000000
c_compression        0.000000
c_range_1h_bps       0.000000
c_range_30m_bps      0.000000
c_vol_1h             0.000000
c_vol_30m            0.000000
c_absret_1h          0.000000
c_absret_30m         0.000000
c_absret_15m         0.000000
close                0.000000
dtype: float64


Features avec > 80% NaN:


Series([], dtype: float64)

In [ ]:
# Cell 11 — Mini test “pipeline-like” : construire y sur un petit sample (sans fit)
# Ce test ne peut marcher que si mfe_R / mae_R existent déjà dans le parquet
if "mfe_R" in df2.columns and "mae_R" in df2.columns:
    tp_rr = 1.3
    max_pullback_R = 0.6
    mfe = df2["mfe_R"].to_numpy(dtype=float)
    mae = df2["mae_R"].to_numpy(dtype=float)
    y = ((mfe >= tp_rr) & (mae <= max_pullback_R)).astype(np.int8)
    print("base rate y=1:", y.mean(), "n:", len(y))
else:
    print("mfe_R/mae_R absents (normal pour events_all_base). ExpertB_light devra les calculer à partir des candles 1m.")